In [1]:
#Celda 1 — Imports y configuración:

import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()

BCRP_BASE = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"

SERIES = {
    "tipo_cambio_venta": "PD04638PD",
    "tipo_cambio_compra": "PD04637PD",
    "tasa_interbancaria": "PD04649PD",
}

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [2]:
# Celda 2 — Función para fechas:

def get_fecha_rango(dias=30):
    hoy = datetime.today()
    inicio = hoy - timedelta(days=dias)
    return inicio.strftime("%Y-%m-%d"), hoy.strftime("%Y-%m-%d")

fecha_inicio, fecha_fin = get_fecha_rango()
print(f"Rango: {fecha_inicio} → {fecha_fin}")

Rango: 2026-03-16 → 2026-04-15


In [3]:
# Celda 3 — Función para traer datos del BCRP:

import locale

MESES_ES = {
    "Ene": "Jan", "Feb": "Feb", "Mar": "Mar", "Abr": "Apr",
    "May": "May", "Jun": "Jun", "Jul": "Jul", "Ago": "Aug",
    "Sep": "Sep", "Oct": "Oct", "Nov": "Nov", "Dic": "Dec"
}

def convertir_fecha_bcrp(fecha_str):
    """Convierte '01.Abr.26' → datetime"""
    try:
        partes = fecha_str.split(".")
        dia = partes[0]
        mes = MESES_ES.get(partes[1], partes[1])
        anio = "20" + partes[2]
        return pd.to_datetime(f"{dia} {mes} {anio}", format="%d %b %Y")
    except Exception:
        return None

def fetch_serie(codigo, fecha_inicio, fecha_fin):
    url = f"{BCRP_BASE}/{codigo}/json/{fecha_inicio}/{fecha_fin}/esp"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        periodos = data.get("periods", [])
        registros = []
        for p in periodos:
            fecha_raw = p.get("name")
            valor = p.get("values", [None])[0]
            if valor is not None:
                try:
                    fecha = convertir_fecha_bcrp(fecha_raw)
                    if fecha is not None:
                        registros.append({
                            "fecha": fecha,
                            "valor": float(valor)
                        })
                except (ValueError, TypeError):
                    pass
        df = pd.DataFrame(registros)
        if not df.empty:
            df = df.sort_values("fecha").reset_index(drop=True)
        return df
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

print("Función corregida lista")

Función corregida lista


In [4]:
#Celda 4 — Trae el tipo de cambio y muéstralo:

df_tc = fetch_serie("PD04638PD", fecha_inicio, fecha_fin)
print(f"Filas obtenidas: {len(df_tc)}")
df_tc.tail(10)

Filas obtenidas: 20


,fecha,valor
10,2026-03-30,3.490714
11,2026-03-31,3.490143
12,2026-04-01,3.459000
13,2026-04-06,3.429643
14,2026-04-07,3.432500
15,2026-04-08,3.387143
16,2026-04-09,3.382143
17,2026-04-10,3.388000
18,2026-04-13,3.370929
19,2026-04-14,3.391857


In [5]:
# Celda 5 — Calcula variaciones:

ultimo = df_tc.iloc[-1]
anterior = df_tc.iloc[-2]
hace_7d = df_tc.iloc[-7]

diff_1d = ultimo["valor"] - anterior["valor"]
diff_7d = ultimo["valor"] - hace_7d["valor"]

print(f"Tipo de cambio hoy ({ultimo['fecha'].strftime('%d/%m/%Y')}): S/ {ultimo['valor']:.4f}")
print(f"vs ayer:   {'+' if diff_1d >= 0 else ''}{diff_1d:.4f} ({diff_1d/anterior['valor']*100:.2f}%)")
print(f"vs 7 días: {'+' if diff_7d >= 0 else ''}{diff_7d:.4f} ({diff_7d/hace_7d['valor']*100:.2f}%)")

Tipo de cambio hoy (14/04/2026): S/ 3.3919
vs ayer:   +0.0209 (0.62%)
vs 7 días: -0.0378 (-1.10%)


In [10]:
SERIES_COMPLETAS = {
    "tipo_cambio_venta":  "PD04638PD",
    "tipo_cambio_compra": "PD04637PD",
    "tasa_interbancaria": "PD04649PD",
    "inflacion_mensual":  "PN01270PM",
    "tasa_referencia":    "PN00015MM",
}

print(SERIES_COMPLETAS)

{'tipo_cambio_venta': 'PD04638PD', 'tipo_cambio_compra': 'PD04637PD', 'tasa_interbancaria': 'PD04649PD', 'inflacion_mensual': 'PN01270PM', 'tasa_referencia': 'PN00015MM'}


In [12]:
def fetch_todas_las_series(series_dict, fecha_inicio, fecha_fin):
    resultados = {}
    for nombre, codigo in series_dict.items():
        df = fetch_serie(codigo, fecha_inicio, fecha_fin)
        resultados[nombre] = df
        if not df.empty:
            ultimo_valor = df.iloc[-1]["valor"]
            ultima_fecha = df.iloc[-1]["fecha"].strftime("%d/%m/%Y")
            print(f"{nombre}: {ultimo_valor} ({ultima_fecha})")
        else:
            print(f"{nombre}: sin datos")
    return resultados

In [13]:
datos = fetch_todas_las_series(SERIES_COMPLETAS, fecha_inicio, fecha_fin)

tipo_cambio_venta: 3.39185714285714 (14/04/2026)
tipo_cambio_compra: 3.38542857142857 (14/04/2026)
Error: 403 Client Error: Forbidden for url: https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PD04649PD/json/2026-03-16/2026-04-15/esp
tasa_interbancaria: sin datos
inflacion_mensual: sin datos
tasa_referencia: sin datos


In [14]:
SERIES_COMPLETAS = {
    "tipo_cambio_venta":  "PD04638PD",
    "tipo_cambio_compra": "PD04637PD",
    "inflacion_mensual":  "PN01270PM",
    "tasa_referencia":    "PN00015MM",
}

fecha_inicio_mensual = "2025-12-01"
fecha_fin_mensual = fecha_fin

datos_diarios = fetch_todas_las_series(
    {"tipo_cambio_venta": "PD04638PD",
     "tipo_cambio_compra": "PD04637PD"},
    fecha_inicio, fecha_fin
)

datos_mensuales = fetch_todas_las_series(
    {"inflacion_mensual": "PN01270PM",
     "tasa_referencia": "PN00015MM"},
    fecha_inicio_mensual, fecha_fin_mensual
)

tipo_cambio_venta: 3.39185714285714 (14/04/2026)
tipo_cambio_compra: 3.38542857142857 (14/04/2026)
inflacion_mensual: sin datos
tasa_referencia: sin datos


In [15]:
url_prueba = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PN01270PM/json/2025-01-01/2026-04-15/esp"
import requests
r = requests.get(url_prueba, timeout=10)
print(r.status_code)
print(r.text[:500])

200
{
"config":
{
"title":"Índice de precios Lima Metropolitana (índice 2009 = 100) (descontinuada)",
"series":
[
{
"name":"Índice de precios Lima Metropolitana (índice 2009 = 100) (descontinuada) - Índice de Precios al Consumidor (IPC)",
"dec":"2"
}]
},
"periods":
[
]
}


In [16]:
url_prueba2 = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PN01273PM/json/2025-01-01/2026-04-15/esp"
r2 = requests.get(url_prueba2, timeout=10)
print(r2.status_code)
print(r2.text[:500])

200
{
"config":
{
"title":"Índice de precios Lima Metropolitana (var% 12 meses)",
"series":
[
{
"name":"Índice de precios Lima Metropolitana (var% 12 meses) - IPC",
"dec":"2"
}]
},
"periods":
[
{
"name":"Ene.2025",
"values":["1.85158283440667"]
},
{
"name":"Feb.2025",
"values":["1.4773290589483"]
},
{
"name":"Mar.2025",
"values":["1.27718026977924"]
},
{
"name":"Abr.2025",
"values":["1.65183401740204"]
},
{
"name":"May.2025",
"values":["1.68666289219268"]
},
{
"name":"Jun.2025",
"values":["1.6937090


In [19]:
SERIES_COMPLETAS = {
    "tipo_cambio_venta":  "PD04638PD",
    "tipo_cambio_compra": "PD04637PD",
    "inflacion_12meses":  "PN01273PM",
    "tasa_referencia":    "PN00015MM",
}

datos_diarios = fetch_todas_las_series(
    {
        "tipo_cambio_venta":  "PD04638PD",
        "tipo_cambio_compra": "PD04637PD"
    },
    fecha_inicio, fecha_fin
)

datos_mensuales = fetch_todas_las_series(
    {
        "inflacion_12meses": "PN01273PM",
        "tasa_referencia":   "PN00015MM"
    },
    "2025-01-01", fecha_fin
)

tipo_cambio_venta: 3.39185714285714 (14/04/2026)
tipo_cambio_compra: 3.38542857142857 (14/04/2026)
inflacion_12meses: 3.80454581291585 (01/03/2026)
tasa_referencia: 28910.221602 (01/02/2026)


In [18]:
def convertir_fecha_bcrp(fecha_str):
    MESES_ES = {
        "Ene": "Jan", "Feb": "Feb", "Mar": "Mar", "Abr": "Apr",
        "May": "May", "Jun": "Jun", "Jul": "Jul", "Ago": "Aug",
        "Sep": "Sep", "Oct": "Oct", "Nov": "Nov", "Dic": "Dec"
    }
    try:
        partes = fecha_str.split(".")
        if len(partes) == 3:
            dia = partes[0]
            mes = MESES_ES.get(partes[1], partes[1])
            anio = "20" + partes[2]
            return pd.to_datetime(f"{dia} {mes} {anio}", format="%d %b %Y")
        elif len(partes) == 2:
            mes = MESES_ES.get(partes[0], partes[0])
            anio = partes[1]
            return pd.to_datetime(f"01 {mes} {anio}", format="%d %b %Y")
    except Exception:
        return None

In [20]:
url_tasa = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PN01234PM/json/2025-01-01/2026-04-15/esp"
r3 = requests.get(url_tasa, timeout=10)
print(r3.status_code)
print(r3.text[:500])

200
{
"config":
{
"title":"Tipo de cambio de las principales monedas - promedio del período (S/ por UM)",
"series":
[
{
"name":"Tipo de cambio de las principales monedas - promedio del período (S/ por UM) - Dólar Americano (US$)",
"dec":"3"
}]
},
"periods":
[
{
"name":"Ene.2025",
"values":["3.74775"]
},
{
"name":"Feb.2025",
"values":["3.6977"]
},
{
"name":"Mar.2025",
"values":["3.65259523809524"]
},
{
"name":"Abr.2025",
"values":["3.6997"]
},
{
"name":"May.2025",
"values":["3.66014285714286"]
},
{
"


In [21]:
url_tasa2 = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PN07819NM/json/2025-01-01/2026-04-15/esp"
r4 = requests.get(url_tasa2, timeout=10)
print(r4.status_code)
print(r4.text[:500])

200
{
"config":
{
"title":"Tasas de interés activas y pasivas promedio de las empresas bancarias en MN (términos efectivos anuales)",
"series":
[
{
"name":"Tasas de interés activas y pasivas promedio de las empresas bancarias en MN (términos efectivos anuales) - Tasa Interbancaria Promedio",
"dec":"1"
}]
},
"periods":
[
{
"name":"Ene.2025",
"values":["4.7349"]
},
{
"name":"Feb.2025",
"values":["4.7755"]
},
{
"name":"Mar.2025",
"values":["4.7203"]
},
{
"name":"Abr.2025",
"values":["4.7565"]
},
{
"nam


In [22]:
SERIES_MENSUALES = {
    "inflacion_12meses":  "PN01273PM",
    "tasa_interbancaria": "PN07819NM",
}

datos_mensuales = fetch_todas_las_series(
    SERIES_MENSUALES,
    "2025-01-01", fecha_fin
)

inflacion_12meses: 3.80454581291585 (01/03/2026)
tasa_interbancaria: 4.2473 (01/03/2026)
